In [ ]:
from __future__ import annotations

import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
def load_report_data(data_dir: str, metric: str) -> pd.DataFrame:
    """Load CSV report files from a model directory and return a tidy DataFrame.

    Parameters
    ----------
    data_dir : str
        Path to the model report folder (e.g. ``report/gpt41mini``).
    metric : str
        Column name to extract (``"accuracy"``, ``"correctness"``, or ``"completeness"``).

    Returns
    -------
    pd.DataFrame
        Columns: ``assay``, ``condition``, ``metric_value``, ``mean``, ``min``, ``max``.
    """
    data_path = Path(data_dir)
    model_name = data_path.name
    csv_files = sorted(data_path.glob("*.csv"))

    rows: list[dict[str, object]] = []
    for csv_file in csv_files:
        stem = csv_file.stem  # e.g. 2026-02-28-report-imc-2d-gpt41mini-baseline
        # Strip date prefix: remove everything before and including "-report-"
        middle = stem.split("-report-", 1)[1]  # e.g. imc-2d-gpt41mini-baseline
        # Split on "-{model_name}-" from the right to handle hyphenated assay names
        suffix = f"-{model_name}-"
        idx = middle.rfind(suffix)
        assay = middle[:idx]  # e.g. imc-2d
        condition = middle[idx + len(suffix) :]  # e.g. baseline

        df = pd.read_csv(csv_file)
        for val in df[metric]:
            rows.append({"assay": assay, "condition": condition, "metric_value": val})

    result = pd.DataFrame(rows)

    # Compute summary stats per assay per condition
    stats = (
        result.groupby(["assay", "condition"])["metric_value"]
        .agg(["mean", "min", "max"])
        .reset_index()
    )
    result = result.merge(stats, on=["assay", "condition"])
    return result

In [ ]:
def plot_grouped_bar_chart(data_dir: str, metric: str) -> None:
    """Grouped bar chart with mean height and min-max error bars."""
    data_path = Path(data_dir)
    model_name = data_path.name
    df = load_report_data(data_dir, metric)

    stats = (
        df.groupby(["assay", "condition"])["metric_value"]
        .agg(["mean", "min", "max"])
        .reset_index()
    )
    assays = sorted(stats["assay"].unique())
    x = np.arange(len(assays))
    width = 0.35

    fig, ax = plt.subplots(figsize=(7, 3.5))
    for i, (condition, color) in enumerate(
        [("baseline", "#4472C4"), ("experiment", "#ED7D31")]
    ):
        cond_stats = (
            stats[stats["condition"] == condition].set_index("assay").reindex(assays)
        )
        means = cond_stats["mean"].values
        err_low = means - cond_stats["min"].values
        err_high = cond_stats["max"].values - means
        ax.bar(
            x + (i - 0.5) * width,
            means,
            width,
            yerr=[err_low, err_high],
            capsize=3,
            color=color,
            label=condition.capitalize(),
        )

    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel(metric.capitalize())
    ax.set_xticks(x)
    ax.set_xticklabels(assays, rotation=45, ha="right")
    ax.set_title(
        f"{model_name} \u2014 {metric.capitalize()} by Assay (Baseline vs Experiment)"
    )
    ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

In [ ]:
def plot_box_plot(data_dir: str, metric: str) -> None:
    """Side-by-side box plots per assay."""
    data_path = Path(data_dir)
    model_name = data_path.name
    df = load_report_data(data_dir, metric)

    assays = sorted(df["assay"].unique())
    colors = {"baseline": "#4472C4", "experiment": "#ED7D31"}

    fig, ax = plt.subplots(figsize=(7, 3.5))
    tick_positions = []
    bp_data_baseline = []
    bp_data_experiment = []

    for i, assay in enumerate(assays):
        tick_positions.append(i * 3 + 0.5)
        bp_data_baseline.append(
            df[(df["assay"] == assay) & (df["condition"] == "baseline")][
                "metric_value"
            ].values
        )
        bp_data_experiment.append(
            df[(df["assay"] == assay) & (df["condition"] == "experiment")][
                "metric_value"
            ].values
        )

    pos_baseline = [i * 3 for i in range(len(assays))]
    pos_experiment = [i * 3 + 1 for i in range(len(assays))]

    bp1 = ax.boxplot(
        bp_data_baseline,
        positions=pos_baseline,
        widths=0.8,
        patch_artist=True,
        boxprops=dict(facecolor=colors["baseline"]),
        medianprops=dict(color="white"),
        whis=[0, 100],
    )
    bp2 = ax.boxplot(
        bp_data_experiment,
        positions=pos_experiment,
        widths=0.8,
        patch_artist=True,
        boxprops=dict(facecolor=colors["experiment"]),
        medianprops=dict(color="white"),
        whis=[0, 100],
    )

    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel(metric.capitalize())
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(assays, rotation=45, ha="right")
    ax.set_title(
        f"{model_name} \u2014 {metric.capitalize()} by Assay (Baseline vs Experiment)"
    )
    ax.legend(
        [bp1["boxes"][0], bp2["boxes"][0]],
        ["Baseline", "Experiment"],
        loc="upper right",
    )
    fig.tight_layout()
    plt.show()

In [ ]:
def plot_dot_range(data_dir: str, metric: str) -> None:
    """Dot-and-range plot: mean as dot, vertical line for min-max range."""
    data_path = Path(data_dir)
    model_name = data_path.name
    df = load_report_data(data_dir, metric)

    stats = (
        df.groupby(["assay", "condition"])["metric_value"]
        .agg(["mean", "min", "max"])
        .reset_index()
    )
    assays = sorted(stats["assay"].unique())
    colors = {"baseline": "#4472C4", "experiment": "#ED7D31"}
    offsets = {"baseline": -0.15, "experiment": 0.15}

    fig, ax = plt.subplots(figsize=(7, 3.5))
    x = np.arange(len(assays))

    for condition in ["baseline", "experiment"]:
        cond_stats = (
            stats[stats["condition"] == condition].set_index("assay").reindex(assays)
        )
        offset = offsets[condition]
        ax.vlines(
            x + offset,
            cond_stats["min"].values,
            cond_stats["max"].values,
            color=colors[condition],
            linewidth=2,
        )
        ax.scatter(
            x + offset,
            cond_stats["mean"].values,
            color=colors[condition],
            s=40,
            zorder=5,
            label=condition.capitalize(),
        )

    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel(metric.capitalize())
    ax.set_xticks(x)
    ax.set_xticklabels(assays, rotation=45, ha="right")
    ax.set_title(
        f"{model_name} \u2014 {metric.capitalize()} by Assay (Baseline vs Experiment)"
    )
    ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

In [ ]:
# gpt41mini — Accuracy
plot_grouped_bar_chart("report/gpt41mini", "accuracy")
plot_box_plot("report/gpt41mini", "accuracy")
plot_dot_range("report/gpt41mini", "accuracy")

In [ ]:
# gpt5mini — Accuracy
plot_grouped_bar_chart("report/gpt5mini", "accuracy")
plot_box_plot("report/gpt5mini", "accuracy")
plot_dot_range("report/gpt5mini", "accuracy")